In [22]:
# ── 환경 변수 로드 ──────────────────────────────────────────
# .env 파일에 저장된 API 키(OpenAI, Tavily 등)를 환경 변수로 불러옵니다.
# load_dotenv()를 호출해야 os.getenv() 또는 라이브러리 내부에서 키를 읽을 수 있습니다.
from dotenv import load_dotenv

load_dotenv(r"C:\Users\pc\Desktop\src\.env")


True

In [23]:
# ── LangSmith 추적 설정 ──────────────────────────────────────
# LangSmith는 LangChain 실행 흐름을 시각적으로 모니터링해 주는 디버깅 도구입니다.
# langsmith.langchain.com 에서 토큰 사용량, 중간 단계, 에러를 확인할 수 있습니다.
# !pip install -qU langchain-teddynote
from langchain_teddynote import logging

# 프로젝트 이름을 입력합니다.
logging.langsmith("CH15-Agentic-RAG")


LangSmith 추적을 시작합니다.
[프로젝트명]
CH15-Agentic-RAG


In [24]:
# ── 패키지 설치 ──────────────────────────────────────────────
# numpy는 Python 3.12에서 소스 빌드 오류가 발생할 수 있어 바이너리로만 설치합니다.
# --only-binary=:all: 옵션은 소스 컴파일 없이 미리 빌드된 wheel 파일만 사용하도록 강제합니다.
!pip install numpy --only-binary=:all:
!pip install langchain langchain-community langchain-core langchain-openai langchain-text-splitters --upgrade


In [25]:
# ── Tavily 웹 검색 도구 설정 ──────────────────────────────────
# TavilySearchResults: 실시간 인터넷 검색 결과를 반환하는 도구입니다.
# 에이전트가 최신 정보(PDF에 없는 정보)를 찾을 때 자동으로 이 도구를 선택합니다.
#
# k=6: 검색 결과를 최대 6개까지 가져옵니다. (k가 클수록 더 많은 맥락 제공)
from langchain_community.tools.tavily_search import TavilySearchResults

search = TavilySearchResults(k=6)


In [26]:
# ── Tavily 검색 테스트 ────────────────────────────────────────
# invoke()로 직접 검색 테스트. 실제로 웹에서 정보를 가져오는지 확인합니다.
search.invoke("뚜기 카카오 닷컴의 사이트에서 전화번호는 뭐입니까?")


Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')


[{'title': "판교 아지트 1층 카카오프렌즈샵 (카카오프렌즈 판교아지트점) - ss's log",
  'url': 'https://sshong.kr/1641',
  'content': "반응형\n\n저작자표시  (새창열림)\n\n#### '생활정보' 카테고리의 다른 글\n\n|  |  |\n --- |\n| 고척스카이돔 경기장 근처 유료주차장 - 귀뚜라미 크린테니스코트 주차장 (일주차 18000원)  (0) | 2022.11.28 |\n| LG 올레드TV (OLED77C1KNB) 무상수리 만료전 전체점검 & 매직리모콘 불량으로 교환받음  (0) | 2022.11.19 |\n| 홍천 장원막국수 비빔막국수 / 서현 장원막국수 들기름막국수  (0) | 2022.11.19 |\n| 갤럭시워치5 블랙 - 심슨 스트랩 워치5락실 쿠폰으로 구매  (0) | 2022.10.27 |\n| 갤럭시 워치5락실 액세서리 쿠폰 사용기간 11월 30일까지  (0) | 2022.10.25 |\n| 수면 케어솔루션 brid.zzz (브릿.지지지) 체험단 당첨  (2) | 2022.10.08 |\n| Baseus(베이스어스) C타입 멀티허브 8in1  (0) | 2022.09.30 |\n\n## 태그\n\n카카오본사카카오프렌즈샵, 카카오프렌즈매장, 카카오프렌즈샵, 카카오프렌즈오프라인, 카카오프렌즈오프매장, 카카오프렌즈판교, 판교아지트, 판교카카오아지트, 판교카카오프렌즈샵, 판교프렌즈샵\n\n## 관련글\n\n   LG 올레드TV (OLED77C1KNB) 무상수리 만료전 전체점검 & 매직리모콘 불량으로 교환받음\n   홍천 장원막국수 비빔막국수 / 서현 장원막국수 들기름막국수\n   갤럭시워치5 블랙 - 심슨 스트랩 워치5락실 쿠폰으로 구매\n   갤럭시 워치5락실 액세서리 쿠폰 사용기간 11월 30일까지\n\n## 댓글0\n\n## 티스토리툴바 [...] 본문 바로가기\n\n# ss's log\n\n생활정보\n\n# 판교 아지트 1층 카카오프렌즈샵 (카카오프렌즈 판교아지트점)\n\n

In [27]:
# ── PDF 로드 → 청크 분할 → 벡터 스토어 생성 ──────────────────
# 1) PyPDFLoader: PDF를 페이지 단위로 로드합니다.
# 2) RecursiveCharacterTextSplitter:
#    - chunk_size=1000: 각 청크의 최대 글자 수
#    - chunk_overlap=100: 청크 간 겹치는 글자 수 (문맥 손실 방지)
#    → 큰 문서를 LLM이 처리 가능한 작은 조각으로 나눕니다.
# 3) FAISS: Facebook에서 만든 벡터 유사도 검색 라이브러리
#    - 텍스트를 임베딩 벡터로 변환 후 저장
#    - 질문과 가장 유사한 청크를 빠르게 검색합니다.
# 4) as_retriever(): FAISS를 LangChain Retriever 인터페이스로 감쌉니다.
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings
from langchain_community.document_loaders import PyPDFLoader

# PDF 파일 로드. 경로는 환경에 맞게 수정
loader = PyPDFLoader("data/SPRI_AI_Brief_2023년12월호_F.pdf")

# 텍스트 분할기를 생성하여 청크로 나눕니다.
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100)

# 문서를 로드하고 분할합니다.
split_docs = loader.load_and_split(text_splitter)

# VectorStore를 생성합니다.
vector = FAISS.from_documents(split_docs, OpenAIEmbeddings())

# Retriever를 생성합니다.
retriever = vector.as_retriever()


Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')


In [28]:
# ── Retriever 단독 테스트 ─────────────────────────────────────
# invoke()로 PDF 내에서 관련 청크를 제대로 찾는지 확인합니다.
# 이 단계는 에이전트 없이 RAG의 '검색' 부분만 테스트하는 것입니다.
retriever.invoke("삼성전자가 자체 개발한 생성형 AI 개발 모델의 상세정보를 찾아줘")


[Document(id='9e60c46c-6697-4639-a309-5f30de9ceaa9', metadata={'producer': 'Hancom PDF 1.3.0.542', 'creator': 'Hwp 2018 10.0.0.13462', 'creationdate': '2023-12-08T13:28:38+09:00', 'author': 'dj', 'moddate': '2023-12-08T13:28:38+09:00', 'pdfversion': '1.4', 'source': 'data/SPRI_AI_Brief_2023년12월호_F.pdf', 'total_pages': 23, 'page': 12, 'page_label': '13'}, page_content='SPRi AI Brief |  2023-12월호\n10\n삼성전자, 자체 개발 생성 AI ‘삼성 가우스’ 공개n삼성전자가 온디바이스에서 작동 가능하며 언어, 코드, 이미지의 3개 모델로 구성된 자체 개발 생성 AI 모델 ‘삼성 가우스’를 공개n삼성전자는 삼성 가우스를 다양한 제품에 단계적으로 탑재할 계획으로, 온디바이스 작동이 가능한 삼성 가우스는 외부로 사용자 정보가 유출될 위험이 없다는 장점을 보유\nKEY Contents'),
 Document(id='e6f597ab-91d8-4255-94e6-21f08df22b97', metadata={'producer': 'Hancom PDF 1.3.0.542', 'creator': 'Hwp 2018 10.0.0.13462', 'creationdate': '2023-12-08T13:28:38+09:00', 'author': 'dj', 'moddate': '2023-12-08T13:28:38+09:00', 'pdfversion': '1.4', 'source': 'data/SPRI_AI_Brief_2023년12월호_F.pdf', 'total_pages': 23, 'page': 12, 'page_label': '13'}, page_content='KEY Contents\n

In [29]:
# ── Retriever를 에이전트 도구로 변환 ──────────────────────────
# create_retriever_tool(): Retriever를 에이전트가 사용할 수 있는 Tool 객체로 변환합니다.
#
# [핵심] name과 description이 중요한 이유:
#   에이전트는 질문을 보고 어떤 도구를 써야 할지 스스로 판단합니다.
#   이때 판단 근거가 바로 description입니다.
#   description이 부정확하면 에이전트가 잘못된 도구를 선택할 수 있습니다.
#
# 예: "PDF에 있는 AI 브리핑 내용"을 질문하면 → pdf_search 선택
#     "최신 뉴스"를 질문하면 → TavilySearchResults 선택
from langchain_core.tools.retriever import create_retriever_tool


retriever_tool = create_retriever_tool(
    retriever,
    name="pdf_search",  # 도구의 이름을 입력합니다.
    description="use this tool to search information from the PDF document",  # 도구의 설명을 자세히 작성해야 합니다!!
)


In [30]:
# ── 에이전트에 등록할 도구 목록 ──────────────────────────────
# 에이전트는 이 tools 리스트 안에서만 도구를 선택할 수 있습니다.
# - search: 실시간 웹 검색 (Tavily)
# - retriever_tool: PDF 문서 검색 (FAISS + RAG)
tools = [search, retriever_tool]


In [32]:
# ── (참고) 버전 충돌 시 재설치 명령어 ───────────────────────────
# 에이전트 실습에서 버전 불일치 문제가 발생하면 아래 명령어로 재설치하세요.
# !pip install numpy --only-binary=:all:
# !pip install langchain==0.3.0 langchain-core==0.3.0 langchain-community==0.3.0 --no-deps
# !pip install langchain-openai langchain-text-splitters


In [33]:
# ── (참고) 특정 버전 고정 설치 ───────────────────────────────
# 교재가 langchain 0.3.x 기준으로 작성된 경우 아래 버전으로 맞출 수 있습니다.
# !pip install "langchain==0.3.25" "langchain-core==0.3.55" "langchain-community==0.3.21" --only-binary=:all:


In [34]:
# ── (참고) 최신 버전 재설치 (권장) ──────────────────────────
# 버전 충돌로 에이전트 실행이 안 될 때, 아래 명령어로 최신 안정 버전을 설치합니다.
# !pip install "numpy>=1.26.4" --only-binary=:all:
# !pip install langchain==0.3.7 langchain-core==0.3.15 langchain-community==0.3.7 langchain-openai==0.2.14 --only-binary=:all:


In [35]:
# ── ReAct 에이전트 생성 (LangGraph 방식) ──────────────────────
# [ReAct 패턴이란?]
#   Reasoning(추론) + Acting(행동)의 반복 루프입니다.
#   에이전트가 "생각 → 도구 호출 → 관찰 → 다시 생각" 과정을 반복하며
#   최종 답변에 도달합니다.
#
# [langchain 1.x 변경점]
#   구버전: AgentExecutor + create_openai_tools_agent 조합
#   신버전: create_react_agent (LangGraph 기반) 하나로 통합
#
# [MemorySaver란?]
#   대화 히스토리를 메모리에 저장합니다.
#   thread_id를 이용해 여러 세션을 독립적으로 관리합니다.
#   같은 thread_id = 이전 대화 기억 / 다른 thread_id = 새 대화 시작
from langchain_openai import ChatOpenAI
from langgraph.prebuilt import create_react_agent
from langgraph.checkpoint.memory import MemorySaver
from langchain_core.messages import HumanMessage

# LLM 모델 설정
llm = ChatOpenAI(model="gpt-4o", temperature=0)

# LangGraph 메모리 설정
memory = MemorySaver()

# Agent 생성 (LangGraph 방식 - langchain 1.x 호환)
agent_executor = create_react_agent(llm, tools, checkpointer=memory)
print("Agent 생성 완료")


Agent 생성 완료


C:\Users\pc\AppData\Local\Temp\ipykernel_65740\1278311266.py:10: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent_executor = create_react_agent(llm, tools, checkpointer=memory)


In [36]:
# ── 에이전트 준비 확인 ────────────────────────────────────────
# agent_executor는 위 셀에서 이미 생성되었습니다.
# 이 셀은 단순히 준비 상태를 확인하는 용도입니다.
print("agent_executor 준비 완료")


agent_executor 준비 완료


In [37]:
# ── AgentStreamParser: 스트리밍 출력 파싱 도구 ───────────────
# 에이전트의 스트리밍 응답을 단계별로 파싱해 보기 좋게 출력합니다.
# 에이전트가 어떤 도구를 선택하고, 어떤 입력/출력을 주고받는지 확인할 수 있습니다.
from langchain_teddynote.messages import AgentStreamParser

# 각 단계별 출력을 위한 파서 생성
agent_stream_parser = AgentStreamParser()


In [38]:
# ── 웹 검색 테스트 (히스토리 없음) ───────────────────────────
# thread_id="no_history_1": 처음 사용하는 새 세션입니다.
# 이 질문은 최신 야구 정보 → PDF에 없는 내용 → 에이전트가 Tavily 웹 검색 선택
#
# [스트리밍 방식 설명]
#   stream_mode="values": 각 단계에서 전체 상태를 반환합니다.
#   step["messages"][-1]: 가장 마지막 메시지(최신 응답)를 가져옵니다.
config = {"configurable": {"thread_id": "no_history_1"}}
result = agent_executor.stream(
    {"messages": [HumanMessage(content="2024년 한국야구 플레이오프 결과를 5팀 이상 검색하여 알려주세요.")]},
    config=config,
    stream_mode="values"
)

for step in result:
    msg = step["messages"][-1]
    if hasattr(msg, "content") and msg.content:
        print(msg.content)


2024년 프로야구 플레이오프 진출한 5개 팀을 검색하여 알려주세요.


Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')
Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')


[{"title": "2024년 프로야구 플레이오프 일정", "url": "https://2000jyc.tistory.com/entry/2024%EB%85%84-%ED%94%84%EB%A1%9C%EC%95%BC%EA%B5%AC-%ED%94%8C%EB%A0%88%EC%9D%B4%EC%98%A4%ED%94%84-%EC%9D%BC%EC%A0%95", "content": "와일드카드 승자와 3위 팀이 맞붙게 되는 본격적인 포스트시즌인 준플레이오프가 시작됩니다.\n\nKT 위즈는 2020년부터 5년 연속으로 포스트시즌 진출에 성공했습니다.\n\n올 시즌 역시 초반부 리그 10위까지 떨어지면서 포스트시즌 진출 전망이 매우 어두웠으나, 순위를 끌어올리기 시작하더니 10월 1일 기준 SSG 랜더스와 공동 5위를 기록하면서 역대 최초 KBO 5위 결정전에 진출했고, 5위 결정전에서 승리를 기록하면서 이번 시즌 포스트시즌의 막차를 타게 됐습니다다.\n\n지난 시즌 2위를 기록하면서 플레이오프에 직행했지만, 이번 포스트시즌은 5위를 기록하면서 와일드카드 결정전 원정팀의 불리함을 안고 시작하게 됐습니다.\n\n이번 와일드카드 결정전에서 두산과의 2경기 중 18이닝 무실점이라는 놀라운 투수력을 자랑하는 팀으로써, 막강한 화력의 공격력을 갖춘 LG와의 준플레이오프 경기가 기대됩니다.\n\nLG 트윈스는 2019년부터 6년 연속으로 포스트시즌 진출에 성공했습니다.\n\n지난 시즌에는 통합 우승을 기록했지만, 이번 시즌에는 기아 타이거즈, 삼성 라이온즈에 밀려 정규시즌을 3위로 마무리하면서 이번 포스트시즌은 준플레이오프부터 일정을 시작하게 되며, 플레이오프를 거쳐 한국시리즈에 진출하게 된다면 2년 연속 한국시리즈 진출을 이루게 됩니다. [...] |  |  |  |  |\n ---  --- |\n| 시리즈 | 일정 | 경기 | 구장 |\n| 준플레이오프 (준PO) | 10월 5일 (토) | 준PO 1차전 | 잠실 |\n| 10월 6일 (일) | 준PO 2차전 | 잠실 |\n| 10월 7일

Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')


2024년 프로야구 플레이오프에 진출한 팀은 다음과 같습니다:

1. **KIA 타이거즈** (정규시즌 1위)
2. **삼성 라이온즈** (정규시즌 2위)
3. **LG 트윈스** (정규시즌 3위)
4. **두산 베어스** (정규시즌 4위)
5. **KT 위즈** (정규시즌 5위)

KT 위즈는 와일드카드 결정전에서 두산 베어스를 이기고 준플레이오프에 진출했습니다. 

자세한 내용은 [여기](https://m.blog.naver.com/qhadldhaus98/223605904423)에서 확인하실 수 있습니다.


`agent_executor` 객체의 `invoke` 메소드를 사용하여, 질문을 입력으로 제공합니다.


In [39]:
# ── PDF 검색 테스트 (히스토리 없음) ───────────────────────────
# thread_id="no_history_2": 이전 세션과 독립된 새 세션입니다.
# 이 질문은 AI 브리핑 PDF 내용 → 에이전트가 pdf_search 도구 선택
config = {"configurable": {"thread_id": "no_history_2"}}
result = agent_executor.stream(
    {"messages": [HumanMessage(content="삼성전자가 자체 개발한 생성형 AI 개발 모델의 상세정보를 찾아주세요.")]},
    config=config,
    stream_mode="values"
)

for step in result:
    msg = step["messages"][-1]
    if hasattr(msg, "content") and msg.content:
        print(msg.content)


삼성전자가 자체 개발한 생성형 AI 관련된 정보를 문서에서 찾아주세요.


Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')


SPRi AI Brief |  2023-12월호
10
삼성전자, 자체 개발 생성 AI ‘삼성 가우스’ 공개n삼성전자가 온디바이스에서 작동 가능하며 언어, 코드, 이미지의 3개 모델로 구성된 자체 개발 생성 AI 모델 ‘삼성 가우스’를 공개n삼성전자는 삼성 가우스를 다양한 제품에 단계적으로 탑재할 계획으로, 온디바이스 작동이 가능한 삼성 가우스는 외부로 사용자 정보가 유출될 위험이 없다는 장점을 보유
KEY Contents

KEY Contents
£언어, 코드, 이미지의 3개 모델로 구성된 삼성 가우스, 온디바이스 작동 지원n삼성전자가 2023년 11월 8일 열린 ‘삼성 AI 포럼 2023’ 행사에서 자체 개발한 생성 AI 모델 ‘삼성 가우스’를 최초 공개∙정규분포 이론을 정립한 천재 수학자 가우스(Gauss)의 이름을 본뜬 삼성 가우스는 다양한 상황에 최적화된 크기의 모델 선택이 가능∙삼성 가우스는 라이선스나 개인정보를 침해하지 않는 안전한 데이터를 통해 학습되었으며, 온디바이스에서 작동하도록 설계되어 외부로 사용자의 정보가 유출되지 않는 장점을 보유∙삼성전자는 삼성 가우스를 활용한 온디바이스 AI 기술도 소개했으며, 생성 AI 모델을 다양한 제품에 단계적으로 탑재할 계획n삼성 가우스는 △텍스트를 생성하는 언어모델 △코드를 생성하는 코드 모델 △이미지를 생성하는 이미지 모델의 3개 모델로 구성∙언어 모델은 클라우드와 온디바이스 대상 다양한 모델로 구성되며, 메일 작성, 문서 요약, 번역 업무의 처리를 지원∙코드 모델 기반의 AI 코딩 어시스턴트 ‘코드아이(code.i)’는 대화형 인터페이스로 서비스를 제공하며 사내 소프트웨어 개발에 최적화∙이미지 모델은 창의적인 이미지를 생성하고 기존 이미지를 원하는 대로 바꿀 수 있도록 지원하며 저해상도 이미지의 고해상도 전환도 지원nIT 전문지 테크리퍼블릭(TechRepublic)은 온디바이스 AI가 주요 기술 트렌드로 부상했다며, 2024년부터 가우스를 탑재한 삼성 스마트폰이 메타의 라마(Llama)2를 탑재한 퀄컴 기

Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')


삼성전자가 자체 개발한 생성형 AI 모델 '삼성 가우스'에 대한 정보는 다음과 같습니다:

1. **모델 구성**: 삼성 가우스는 언어, 코드, 이미지의 3개 모델로 구성되어 있으며, 온디바이스에서 작동할 수 있도록 설계되었습니다. 이는 사용자 정보가 외부로 유출될 위험이 없다는 장점을 가지고 있습니다.

2. **공식 발표**: 삼성전자는 2023년 11월 8일 '삼성 AI 포럼 2023'에서 삼성 가우스를 최초 공개했습니다. 이 모델은 정규분포 이론을 정립한 수학자 가우스의 이름을 따왔습니다.

3. **기능**:
   - **언어 모델**: 메일 작성, 문서 요약, 번역 등의 작업을 지원합니다.
   - **코드 모델**: AI 코딩 어시스턴트 '코드아이(code.i)'를 통해 대화형 인터페이스로 소프트웨어 개발을 지원합니다.
   - **이미지 모델**: 창의적인 이미지를 생성하고 기존 이미지를 수정할 수 있으며, 저해상도 이미지를 고해상도로 변환하는 기능도 제공합니다.

4. **제품 통합 계획**: 삼성전자는 삼성 가우스를 다양한 제품에 단계적으로 탑재할 계획이며, 2024년부터 가우스를 탑재한 삼성 스마트폰이 메타의 라마(Llama)2를 탑재한 퀄컴 기기 및 구글 픽셀과 경쟁할 것으로 예상하고 있습니다.

5. **기타 기능**: 삼성 가우스는 사업계획 및 발표 자료 작성, 제품 이미지 생성, 개인 비서 역할 등 다양한 기능을 수행할 수 있습니다. 또한, 쇼핑 시 최적의 제품 추천 및 사용자 맞춤형 뉴스 구독 기능도 제공할 예정입니다.

이 정보는 삼성전자가 AI 기술을 통해 사용자 경험을 향상시키고, 데이터 보안을 강화하려는 노력을 반영하고 있습니다.


In [40]:
# ── 대화 히스토리 에이전트 설정 ───────────────────────────────
# [구버전 방식 vs 신버전 방식]
#   구버전: RunnableWithMessageHistory(agent_executor, get_session_history, ...)
#          → 별도 히스토리 관리 객체가 필요했음
#
#   신버전: create_react_agent에 checkpointer=MemorySaver()를 전달
#          → agent_executor 자체가 이미 히스토리를 관리함
#          → 따로 래핑할 필요 없이 같은 thread_id만 사용하면 됨
#
# 결론: agent_with_chat_history = agent_executor (완전히 동일한 객체)
agent_with_chat_history = agent_executor  # agent_executor는 이미 메모리를 가집니다.
print("agent_with_chat_history 준비 완료")


agent_with_chat_history 준비 완료


In [41]:
# ── 대화 히스토리 첫 번째 질문 ────────────────────────────────
# thread_id="abc123"로 새 대화 세션 시작
# 이 질문의 답변은 메모리에 저장되어 다음 질문에서도 참조됩니다.
config = {"configurable": {"thread_id": "abc123"}}
result = agent_with_chat_history.stream(
    {"messages": [HumanMessage(content="삼성전자가 자체 개발한 생성형 AI 개발 모델의 상세정보를 찾아주세요.")]},
    config=config,
    stream_mode="values"
)

for step in result:
    msg = step["messages"][-1]
    if hasattr(msg, "content") and msg.content:
        print(msg.content)


삼성전자가 개발한 생성형 AI 관련된 정보를 문서에서 찾아주세요.


Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')


SPRi AI Brief |  2023-12월호
10
삼성전자, 자체 개발 생성 AI ‘삼성 가우스’ 공개n삼성전자가 온디바이스에서 작동 가능하며 언어, 코드, 이미지의 3개 모델로 구성된 자체 개발 생성 AI 모델 ‘삼성 가우스’를 공개n삼성전자는 삼성 가우스를 다양한 제품에 단계적으로 탑재할 계획으로, 온디바이스 작동이 가능한 삼성 가우스는 외부로 사용자 정보가 유출될 위험이 없다는 장점을 보유
KEY Contents

KEY Contents
£언어, 코드, 이미지의 3개 모델로 구성된 삼성 가우스, 온디바이스 작동 지원n삼성전자가 2023년 11월 8일 열린 ‘삼성 AI 포럼 2023’ 행사에서 자체 개발한 생성 AI 모델 ‘삼성 가우스’를 최초 공개∙정규분포 이론을 정립한 천재 수학자 가우스(Gauss)의 이름을 본뜬 삼성 가우스는 다양한 상황에 최적화된 크기의 모델 선택이 가능∙삼성 가우스는 라이선스나 개인정보를 침해하지 않는 안전한 데이터를 통해 학습되었으며, 온디바이스에서 작동하도록 설계되어 외부로 사용자의 정보가 유출되지 않는 장점을 보유∙삼성전자는 삼성 가우스를 활용한 온디바이스 AI 기술도 소개했으며, 생성 AI 모델을 다양한 제품에 단계적으로 탑재할 계획n삼성 가우스는 △텍스트를 생성하는 언어모델 △코드를 생성하는 코드 모델 △이미지를 생성하는 이미지 모델의 3개 모델로 구성∙언어 모델은 클라우드와 온디바이스 대상 다양한 모델로 구성되며, 메일 작성, 문서 요약, 번역 업무의 처리를 지원∙코드 모델 기반의 AI 코딩 어시스턴트 ‘코드아이(code.i)’는 대화형 인터페이스로 서비스를 제공하며 사내 소프트웨어 개발에 최적화∙이미지 모델은 창의적인 이미지를 생성하고 기존 이미지를 원하는 대로 바꿀 수 있도록 지원하며 저해상도 이미지의 고해상도 전환도 지원nIT 전문지 테크리퍼블릭(TechRepublic)은 온디바이스 AI가 주요 기술 트렌드로 부상했다며, 2024년부터 가우스를 탑재한 삼성 스마트폰이 메타의 라마(Llama)2를 탑재한 퀄컴 기

Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')


삼성전자가 개발한 생성형 AI에 대한 정보는 다음과 같습니다:

1. **삼성 가우스**: 삼성전자는 2023년 11월 8일 '삼성 AI 포럼 2023'에서 자체 개발한 생성형 AI 모델인 '삼성 가우스'를 공개했습니다. 이 모델은 언어, 코드, 이미지의 3개 모델로 구성되어 있으며, 온디바이스에서 작동할 수 있도록 설계되었습니다.

2. **온디바이스 작동**: 삼성 가우스는 외부로 사용자 정보가 유출될 위험이 없다는 장점을 가지고 있습니다. 이는 안전한 데이터를 통해 학습되었으며, 다양한 제품에 단계적으로 탑재될 계획입니다.

3. **모델 구성**:
   - **언어 모델**: 메일 작성, 문서 요약, 번역 업무 등을 지원합니다.
   - **코드 모델**: AI 코딩 어시스턴트 '코드아이(code.i)'를 통해 대화형 인터페이스로 서비스를 제공하며, 사내 소프트웨어 개발에 최적화되어 있습니다.
   - **이미지 모델**: 창의적인 이미지를 생성하고 기존 이미지를 원하는 대로 변경할 수 있으며, 저해상도 이미지를 고해상도로 전환하는 기능도 지원합니다.

4. **경쟁 전망**: IT 전문지 테크리퍼블릭은 2024년부터 삼성 가우스를 탑재한 삼성 스마트폰이 메타의 라마(Llama)2를 탑재한 퀄컴 기기 및 구글 어시스턴트를 적용한 구글 픽셀(Pixel)과 경쟁할 것으로 예상하고 있습니다.

이와 같은 정보는 삼성전자가 생성형 AI 기술을 통해 다양한 분야에서 혁신을 이루고자 하는 노력을 보여줍니다.


In [42]:
# ── 대화 히스토리 두 번째 질문 (연속 대화 확인) ───────────────
# 같은 thread_id="abc123"을 사용 → 이전 대화 내용을 기억합니다.
# "방금 대답" 같은 대명사를 쓸 수 있는 것은 히스토리 덕분입니다.
# 만약 다른 thread_id를 쓰면 에이전트는 이전 대화를 기억하지 못합니다.
config = {"configurable": {"thread_id": "abc123"}}
result = agent_with_chat_history.stream(
    {"messages": [HumanMessage(content="방금 대답한 내용을 정리해 주세요.")]},
    config=config,
    stream_mode="values"
)

for step in result:
    msg = step["messages"][-1]
    if hasattr(msg, "content") and msg.content:
        print(msg.content)


이전의 답변을 영어로 번역해 주세요.


Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')


Here is the translation of the previous response into English:

1. **Samsung Gauss**: Samsung Electronics unveiled its generative AI model, 'Samsung Gauss,' at the 'Samsung AI Forum 2023' on November 8, 2023. This model consists of three components: language, code, and image, and is designed to operate on-device.

2. **On-Device Operation**: Samsung Gauss has the advantage of not risking the leakage of user information externally. It has been trained on safe data and is planned to be gradually integrated into various products.

3. **Model Composition**:
   - **Language Model**: Supports tasks such as email writing, document summarization, and translation.
   - **Code Model**: Provides a conversational interface through the AI coding assistant 'Code.i,' optimized for in-house software development.
   - **Image Model**: Capable of generating creative images and modifying existing images as desired, as well as supporting the conversion of low-resolution images to high-resolution.

4. **Co

In [43]:
# ═══════════════════════════════════════════════════════════════
# [전체 파이프라인 요약] Agentic RAG 완전 셋업
# ═══════════════════════════════════════════════════════════════
# 이 셀은 실습 전체를 처음부터 한 번에 실행하는 종합 템플릿입니다.
# 커널 재시작 후 이 셀 하나만 실행해도 에이전트가 준비됩니다.
#
# 전체 흐름:
# 1) 도구 생성: Tavily(웹 검색) + FAISS(PDF 검색)
# 2) LLM: GPT-4o
# 3) 에이전트: create_react_agent (ReAct 패턴 + MemorySaver)
# 4) 스트리밍 파서: AgentStreamParser
#
# [에이전트 선택 로직]
#   질문 → LLM이 판단 → PDF 관련이면 pdf_search, 최신 정보면 Tavily
#   → 결과를 받아 다시 추론 → 최종 답변 생성

# 필요한 모든 import
from langchain_core.prompts import ChatPromptTemplate
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_core.tools.retriever import create_retriever_tool
from langgraph.prebuilt import create_react_agent
from langgraph.checkpoint.memory import MemorySaver
from langchain_core.messages import HumanMessage
from langchain_teddynote.messages import AgentStreamParser

########## 1. 도구 생성 ##########
# Tavily: 실시간 웹 검색 도구
search = TavilySearchResults(k=6)

# PDF 로드 → 청크 분할 → 벡터 스토어 → Retriever 생성
loader = PyMuPDFLoader("data/SPRI_AI_Brief_2023년12월호_F.pdf")
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100)
split_docs = loader.load_and_split(text_splitter)
vector = FAISS.from_documents(split_docs, OpenAIEmbeddings())
retriever = vector.as_retriever()

# Retriever를 에이전트 도구로 변환
retriever_tool = create_retriever_tool(
    retriever,
    name="pdf_search",
    description="use this tool to search information from the PDF document",
)

# 에이전트에 등록할 도구 리스트
tools = [search, retriever_tool]

########## 2. LLM 설정 ##########
llm = ChatOpenAI(model="gpt-4o", temperature=0)

########## 3. Agent 생성 (LangGraph 방식) ##########
# checkpointer=memory: thread_id 기반 대화 히스토리 자동 관리
memory = MemorySaver()
agent_executor = create_react_agent(llm, tools, checkpointer=memory)
agent_with_chat_history = agent_executor  # 동일한 객체 (히스토리 이미 내장)

########## 4. Agent 파서 생성 ##########
agent_stream_parser = AgentStreamParser()
print("Agent 셋업 완료")


Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')


Agent 템플릿 설정 완료


C:\Users\pc\AppData\Local\Temp\ipykernel_65740\188278905.py:36: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent_executor = create_react_agent(llm, tools, checkpointer=memory)


In [44]:
# ── PDF 검색 질문 (thread_id: abc123) ───────────────────────
# PDF에서 구글 텍스트워터마크 정보를 찾습니다.
# 에이전트는 질문 내용을 보고 pdf_search 도구를 선택합니다.
config = {"configurable": {"thread_id": "abc123"}}
result = agent_with_chat_history.stream(
    {"messages": [HumanMessage(content="구글 텍스트워터마크 기술에 대한 정보를 찾아줘")]},
    config=config,
    stream_mode="values"
)

for step in result:
    msg = step["messages"][-1]
    if hasattr(msg, "content") and msg.content:
        print(msg.content)


구글이 앤스로픽에 투자한 금액을 문서에서 찾아줘


Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')


1. 정책/법제  
2. 기업/산업 
3. 기술/연구 
 4. 인력/교육
구글, 앤스로픽에 20억 달러 투자로 생성 AI 협력 강화 
n 구글이 앤스로픽에 최대 20억 달러 투자에 합의하고 5억 달러를 우선 투자했으며, 앤스로픽은 
구글과 클라우드 서비스 사용 계약도 체결
n 3대 클라우드 사업자인 구글, 마이크로소프트, 아마존은 차세대 AI 모델의 대표 기업인 
앤스로픽 및 오픈AI와 협력을 확대하는 추세
KEY Contents
£ 구글, 앤스로픽에 최대 20억 달러 투자 합의 및 클라우드 서비스 제공
n 구글이 2023년 10월 27일 앤스로픽에 최대 20억 달러를 투자하기로 합의했으며, 이 중 5억 
달러를 우선 투자하고 향후 15억 달러를 추가로 투자할 방침
∙구글은 2023년 2월 앤스로픽에 이미 5억 5,000만 달러를 투자한 바 있으며, 아마존도 지난 9월 
앤스로픽에 최대 40억 달러의 투자 계획을 공개
∙한편, 2023년 11월 8일 블룸버그 보도에 따르면 앤스로픽은 구글의 클라우드 서비스 사용을 위해 
4년간 30억 달러 규모의 계약을 체결
∙오픈AI 창업자 그룹의 일원이었던 다리오(Dario Amodei)와 다니엘라 아모데이(Daniela Amodei) 
남매가 2021년 설립한 앤스로픽은 챗GPT의 대항마 ‘클로드(Claude)’ LLM을 개발
n 아마존과 구글의 앤스로픽 투자에 앞서, 마이크로소프트는 차세대 AI 모델의 대표 주자인 오픈
AI와 협력을 확대
∙마이크로소프트는 오픈AI에 앞서 투자한 30억 달러에 더해 2023년 1월 추가로 100억 달러를 
투자하기로 하면서 오픈AI의 지분 49%를 확보했으며, 오픈AI는 마이크로소프트의 애저(Azure) 
클라우드 플랫폼을 사용해 AI 모델을 훈련
£ 구글, 클라우드 경쟁력 강화를 위해 생성 AI 투자 확대
n 구글은 수익률이 높은 클라우드 컴퓨팅 시장에서 아마존과 마이크로소프트를 따라잡고자 생성 AI를 
통한 기업 고객의 클라우드 지출 확대를 위해 AI 투자를 지속

n 구글은 수익률

Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')


구글은 앤스로픽에 최대 20억 달러를 투자하기로 합의했으며, 이 중 5억 달러를 우선 투자했습니다. 향후 15억 달러를 추가로 투자할 계획입니다.


In [45]:
# ── 연속 대화 - 이전 답변 요약 요청 (thread_id: abc123) ──────
# 같은 thread_id 유지 → 바로 전 답변을 기억합니다.
# "방금 대답한 내용"처럼 대명사를 쓸 수 있는 것이 히스토리의 핵심입니다.
config = {"configurable": {"thread_id": "abc123"}}
result = agent_with_chat_history.stream(
    {"messages": [HumanMessage(content="방금 대답한 내용을 정리해 주세요")]},
    config=config,
    stream_mode="values"
)

for step in result:
    msg = step["messages"][-1]
    if hasattr(msg, "content") and msg.content:
        print(msg.content)


이전의 답변을 영어로 번역해 주세요


Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')


Google has agreed to invest up to $2 billion in Anthropic, with an initial investment of $500 million. They plan to invest an additional $1.5 billion in the future.


In [46]:
# ── 새 세션 시작 - 웹 검색 질문 (thread_id: abc456) ──────────
# thread_id가 바뀌면 이전 대화와 완전히 독립된 새 세션입니다.
# 최신 야구 정보 → PDF에 없음 → 에이전트가 Tavily 웹 검색을 선택합니다.
config = {"configurable": {"thread_id": "abc456"}}
result = agent_with_chat_history.stream(
    {"messages": [HumanMessage(content="2024년 한국야구 플레이오프 관련 5팀이상을 검색해서 알려주세요. 한글로 대답하세요")]},
    config=config,
    stream_mode="values"
)

for step in result:
    msg = step["messages"][-1]
    if hasattr(msg, "content") and msg.content:
        print(msg.content)


2024년 프로야구 플레이오프 진출 5개팀을 검색해서 알려주세요. 한글로 답변하세요


Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')
Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')


[{"title": "2024년 프로야구 플레이오프 일정", "url": "https://2000jyc.tistory.com/entry/2024%EB%85%84-%ED%94%84%EB%A1%9C%EC%95%BC%EA%B5%AC-%ED%94%8C%EB%A0%88%EC%9D%B4%EC%98%A4%ED%94%84-%EC%9D%BC%EC%A0%95", "content": "와일드카드 승자와 3위 팀이 맞붙게 되는 본격적인 포스트시즌인 준플레이오프가 시작됩니다.\n\nKT 위즈는 2020년부터 5년 연속으로 포스트시즌 진출에 성공했습니다.\n\n올 시즌 역시 초반부 리그 10위까지 떨어지면서 포스트시즌 진출 전망이 매우 어두웠으나, 순위를 끌어올리기 시작하더니 10월 1일 기준 SSG 랜더스와 공동 5위를 기록하면서 역대 최초 KBO 5위 결정전에 진출했고, 5위 결정전에서 승리를 기록하면서 이번 시즌 포스트시즌의 막차를 타게 됐습니다다.\n\n지난 시즌 2위를 기록하면서 플레이오프에 직행했지만, 이번 포스트시즌은 5위를 기록하면서 와일드카드 결정전 원정팀의 불리함을 안고 시작하게 됐습니다.\n\n이번 와일드카드 결정전에서 두산과의 2경기 중 18이닝 무실점이라는 놀라운 투수력을 자랑하는 팀으로써, 막강한 화력의 공격력을 갖춘 LG와의 준플레이오프 경기가 기대됩니다.\n\nLG 트윈스는 2019년부터 6년 연속으로 포스트시즌 진출에 성공했습니다.\n\n지난 시즌에는 통합 우승을 기록했지만, 이번 시즌에는 기아 타이거즈, 삼성 라이온즈에 밀려 정규시즌을 3위로 마무리하면서 이번 포스트시즌은 준플레이오프부터 일정을 시작하게 되며, 플레이오프를 거쳐 한국시리즈에 진출하게 된다면 2년 연속 한국시리즈 진출을 이루게 됩니다. [...] |  |  |  |  |\n ---  --- |\n| 시리즈 | 일정 | 경기 | 구장 |\n| 준플레이오프 (준PO) | 10월 5일 (토) | 준PO 1차전 | 잠실 |\n| 10월 6일 (일) | 준PO 2차전 | 잠실 |\n| 10월 7일

Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')


2024년 프로야구 플레이오프에 진출한 5개 팀은 다음과 같습니다:

1. KIA 타이거즈
2. 삼성 라이온즈
3. LG 트윈스
4. 두산 베어스
5. KT 위즈

이 팀들이 포스트시즌에 진출하여 경기를 치르게 됩니다.


In [47]:
# ── 연속 대화 - SNS 게시글 작성 요청 (thread_id: abc456) ─────
# 이전에 검색한 야구 결과를 기억하고 SNS 게시글 형태로 변환합니다.
config = {"configurable": {"thread_id": "abc456"}}
result = agent_with_chat_history.stream(
    {"messages": [HumanMessage(content="방금 대답한 내용을 SNS 게시글 형태로 100자 이내로 작성하세요.")]},
    config=config,
    stream_mode="values"
)

for step in result:
    msg = step["messages"][-1]
    if hasattr(msg, "content") and msg.content:
        print(msg.content)


이전의 답변을 SNS 게시글 형태로 100자 내외로 작성하세요.


Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')


2024년 프로야구 플레이오프 진출팀: KIA 타이거즈, 삼성 라이온즈, LG 트윈스, 두산 베어스, KT 위즈! 치열한 경기가 기대됩니다! ⚾️🔥 #KBO #플레이오프


In [48]:
# ── 연속 대화 - 이모지 추가 요청 (thread_id: abc456) ─────────
# 직전에 작성한 게시글 내용을 그대로 이어받아 이모지를 추가합니다.
# → Agentic RAG + 대화 히스토리의 결합 효과를 확인할 수 있습니다.
config = {"configurable": {"thread_id": "abc456"}}
result = agent_with_chat_history.stream(
    {"messages": [HumanMessage(content="방금 대답한 내용에 한국 이모지를 추가하세요.")]},
    config=config,
    stream_mode="values"
)

for step in result:
    msg = step["messages"][-1]
    if hasattr(msg, "content") and msg.content:
        print(msg.content)


이전의 답변에 한국 시리즈 일정을 추가하세요.


Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')


2024년 프로야구 플레이오프 진출팀: KIA 타이거즈, 삼성 라이온즈, LG 트윈스, 두산 베어스, KT 위즈! 한국 시리즈는 10월 21일부터 광주에서 시작됩니다. ⚾️🔥 #KBO #플레이오프 #한국시리즈


Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')
